# Quantitative Researcher API

WorldQuant-style workflow (decision note DEC-017): seed -> single-alpha evaluation -> GA mining from passing seeds -> pool delivery.

In [ ]:
from quant_api.core import DataConfig
from quant_api.research import (
    ResearchConfig, GateCriteria, validate_seed, evaluate_seed,
    mine_seeds, screen_batch, build_spec_sheet, deliver_to_pool,
)

WINDOW = DataConfig(start="2026-07-15", end="2026-08-28")
cfg = ResearchConfig(data=WINDOW, ga_population_size=8, ga_generations=2, ga_seed=7)
print("ok")

In [ ]:
seed = "close - ewma(close, 8)"
canonical = validate_seed(seed)
tear = evaluate_seed(seed, cfg)
print(canonical, "->", tear.verdict, tear.reasons)
print({k: tear.metrics[k] for k in ("net_sharpe", "best_abs_ic", "cost_drag_pct")})

In [ ]:
# Breed from evaluation-passing seeds (WorldQuant-style: seeds enter mining
# only after passing single-alpha evaluation).
bred = mine_seeds(["close - ewma(close, 8)", "ts_returns(close, 8)"], cfg)
print("bred candidates:", bred[:3])

In [ ]:
funnel = screen_batch(bred[:4], cfg, record_trials=False)
print(funnel["funnel"])

In [ ]:
import tempfile
pool_dir = tempfile.mkdtemp(prefix="research_pool_")
survivor = funnel["tear_sheets"][0]
if survivor.verdict == "IN":
    entry = deliver_to_pool(survivor, build_spec_sheet(survivor, config=cfg),
                            source="ga", pool_root=pool_dir)
    print("delivered:", entry.alpha_id, "->", pool_dir)
else:
    print("no IN survivor in this batch:", survivor.reasons)